#### Import statements

In [15]:
import os
import pathlib
import inspect
import pickle
import numpy as np
import pandas as pd
import json
import socket
import os
import pathlib
import inspect
import pickle
import numpy as np
import pandas as pd
import json
import itertools
import functools

hostname = socket.gethostname()

if 'rc.zi.columbia.edu' in hostname:
    from ws.simulate_task import load_sim_params, load_task_def
    from ws.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, matches_template
    from ws.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws') 
else:
    from simulation_whiskers.simulate_task import load_sim_params, load_task_def
    from simulation_whiskers.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, matches_template
    #from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_autoencoder_geometry
    from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('C:\\', 'Users', 'danie' 'Documents', 'code_libraries', 'simulation_whiskers')
    
from analysis_metadata.analysis_metadata import Metadata, increment_dir_name, write_metadata
import time

#### Define parameters, input paths

In [20]:
# Define path to model parameters file: 
ae_params_path=os.path.join(base, 'hyperparams', 'example_autoencoder_hparams.json')


# Define tasks:
task_defs = [
    
    # Task 0:
    [
        functools.partial(matches_template, template={'freq_sh' : 2}),
        functools.partial(matches_template, template={'freq_sh' : 15})
     ],
    
    # Task 1:
    [
        functools.partial(matches_template, template={'time_mov' : 10}),
        functools.partial(matches_template, template={'time_mov' : 17})
     ]
    ]


# Define general variables:
n_files = 1
n_geo_subsamples = 1
sum_inpt=False
xor=True
zscore_data = False
sig_init = 1.0
save_learning = False
chunked_reconstruction_loss = False


# Define simulation parameters:
concavity = [0]
n_whisk = 2
prob_poiss = 1.01
noise_w = 0.3
spread = 'auto'
speed = 2.0
ini_phase_m = 0
ini_phase_spr = 100
delay_time = 0
freq_m = 3.0
freq_std = 0.1
std_reset = 0
t_total = 2
dt = 0.1
dx = 0.01
n_trials_pre = 50
amp = 2
freq_sh = [2, 15]
z1 = [4]
max_rad = 50
n_rad = 4
disp = 4.5
theta = [0]
steps_mov = [10, 17]
rad_vec = [6]
init_position = 0


# Autoencoder parameters:
mdl_type = "autoencoder"
n_hidden = 20
sig_init = 1 
sig_neu = 0.1 
lr = 0.001
beta0 = 0
beta1 = 0
beta_rec = 0
beta_xor = 0
n_epochs = 50
batch_size = 10
beta_sp = 0
beta_pr = 0
p_norm = 2
n_splits = 5
n_predictor_bins = 10
n_predicted_bins = 4


# Compute parameters:
gpu = False


# Do some custom, ad-hoc hyperparameter selection:
#beta_lins=10**np.arange(0, 5, 0.5)
beta_lins = [0]
n_hiddens = [20]
#hparams = [{'beta_rec':x[0], 'n_hidden':x[1]} for x in list(itertools.product(beta_lins, n_hiddens))]
hparams = None


# Output directory:
if 'rc.zi.columbia' in hostname:
    base_output_directory = os.path.join(base, 'results')
else:
    base_output_directory='E:\\simulation_whiskers\\results\\'
run_base_name='run'
sv=True

#### Define dataframe of hyperparameters to iterate over:

In [21]:
simulation_cols = ['concavity', 'n_whisk', 'prob_poiss', 'noise_w', 'spread',
     'speed', 'ini_phase_m', 'ini_phase_spr', 'delay_time', 'freq_m', 'freq_std',
     'std_reset', 't_total', 'dt', 'dx', 'n_trials_pre', 'n_files', 'amp', 'freq_sh',
     'z1', 'max_rad', 'n_rad', 'disp', 'theta', 'steps_mov', 'rad_vec', 'init_position']

autoencoder_cols = ['mdl_type', 'n_hidden', 'sig_init', 'sig_neu', 'lr', 'beta0',
    'beta1', 'beta_rec', 'beta_xor', 'n_epochs', 'batch_size', 'beta_sp', 'p_norm',
    'beta_pr', 'n_splits', 'n_predictor_bins', 'n_predicted_bins']

hparams_df = generate_hparams_df(hparams=hparams, task_defs=task_defs, n_files=n_files, 
     xor=xor, n_geo_subsamples=n_geo_subsamples, zscore_data=zscore_data, 
     save_perf=False, sum_inpt=sum_inpt, chunked_reconstruction_loss=False, 
     save_learning=save_learning, gpu=gpu, save_sessions=False, verbose=False, 
     concavity=concavity, n_whisk=n_whisk, prob_poiss=prob_poiss, noise_w=noise_w, 
     spread=spread, speed=speed, ini_phase_m=ini_phase_m, ini_phase_spr=ini_phase_spr, 
     delay_time=delay_time, freq_m=freq_m, freq_std=freq_std, std_reset=std_reset, 
     t_total=t_total, dt=dt, dx=dx, n_trials_pre=n_trials_pre, n_repeats=n_files, 
     amp=amp, freq_sh=freq_sh, z1=z1, max_rad=max_rad, n_rad=n_rad, disp=disp, 
     theta=theta, steps_mov=steps_mov, rad_vec=rad_vec, init_position=init_position, 
     mdl_type=mdl_type, n_hidden=n_hidden, sig_init=sig_init, sig_neu=sig_neu, 
     lr=lr, beta0=beta0, beta1=beta1, beta_rec=beta_rec, beta_xor=beta_xor, 
     beta_sp=beta_sp, beta_pr=beta_pr, n_epochs=n_epochs, batch_size=batch_size, 
     p_norm=p_norm, n_splits=n_splits, n_predictor_bins=n_predictor_bins, 
     n_predicted_bins=n_predicted_bins)

#### Run whisker simulations, train models:

In [24]:
# Verify parameters before executing:
hparam_strs = list(hparams_df.apply(lambda x : 'model={}, n_hidden={}, beta_sp={}, beta_rec={}, beta_pr={}, n_epochs={}'.format(x.mdl_type,x.n_hidden, x.beta_rec, x.beta_sp, x.beta_pr, x.n_epochs), axis=1))
print('Running following hyperparameters:\n')
print('\n'.join(hparam_strs))
yn = input('\nProceed? (y/n)')
if yn == 'y':
    pass
else: 
    raise AssertionError('User aborted execution.')


# Iterate over dicts of hyperparamter combos:
all_geo_results = pd.DataFrame()
all_perf_results = pd.DataFrame()
all_ae_results = pd.DataFrame()
start = time.time()
for hidx, curr_hparams in hparams_df.iterrows():
    
    curr_sim_params = dict(curr_hparams[simulation_cols])
    curr_autoencoder_params = dict(curr_hparams[autoencoder_cols])
    
    curr_results=mdl_geometry_pipeline(curr_sim_params,  
        tasks=curr_hparams.task_defs, autoencoder_params=curr_autoencoder_params, xor=curr_hparams.xor, 
        n_geo_subsamples=curr_hparams.n_geo_subsamples, zscore_data=curr_hparams.zscore_data, 
        save_perf=False, sum_inpt=curr_hparams.sum_inpt, chunked_reconstruction_loss=curr_hparams.chunked_reconstruction_loss, 
        save_learning=curr_hparams.save_learning, gpu=curr_hparams.gpu, save_sessions=False, 
        verbose=True)

    curr_hparams_df = pd.DataFrame(hparams_df.iloc[0]).T

    # Extract geometry results, add metadata:
    curr_geo_results = curr_results['geo_df']
    geo_meta_cols = list(set(curr_hparams_df) - set(curr_geo_results.columns))
    geo_meta = pd.concat([curr_hparams_df[geo_meta_cols]]*curr_geo_results.shape[0],axis=0)
    geo_meta.index = np.arange(geo_meta.shape[0])
    curr_geo_results = pd.concat([curr_geo_results, geo_meta], axis=1)
    all_geo_results = pd.concat([all_geo_results, curr_geo_results], axis=0)
    
    # Extract classifier performance results, add metadata:
    curr_perf_results = curr_results['perf_df']
    perf_meta_cols = list(set(curr_hparams_df) - set(curr_perf_results.columns))
    perf_meta = pd.concat([curr_hparams_df[perf_meta_cols]]*curr_perf_results.shape[0],axis=0)
    perf_meta.index = np.arange(perf_meta.shape[0])
    curr_perf_results = pd.concat([curr_perf_results, perf_meta], axis=1)
    all_perf_results = pd.concat([all_perf_results, curr_perf_results], axis=0)

    # Extract autoencoder representations, add metadata:
    if curr_results['ae_df'] is not None:            
        curr_ae_results = curr_results['ae_df']
        ae_meta_cols = list(set(curr_hparams_df) - set(curr_ae_results.columns))
        ae_meta = pd.concat([curr_hparams_df[ae_meta_cols]]*curr_ae_results.shape[0],axis=0)
        curr_ae_results = pd.concat([curr_ae_results, perf_meta], axis=1)
        all_ae_results = pd.concat([all_ae_results, curr_ae_results])

all_geo_results['train_partition'] = all_geo_results.apply(lambda x :str(x.train_partition), axis=1)

all_results = dict()
all_results['geo_df'] = all_geo_results
all_results['perf_df'] = all_perf_results
all_results['ae_df'] = all_ae_results

stop = time.time()

Running following hyperparameters:

model=autoencoder, n_hidden=20, beta_sp=0, beta_rec=0, beta_pr=0, n_epochs=50



Proceed? (y/n) y


Simulating whisker contact data...
simulate_session duration=0.615891695022583
Fitting autoencoder...
0 rec  0.2223089039325714 ce  0.0 sp  0.06536365300416946 total  0.0
49 rec  0.22222787141799927 ce  0.0 sp  0.06518704444169998 total  0.0
fit_autoencoder duration=1.8233952522277832


#### Save output:

In [25]:
if sv:
    
    # Save results dataframe:
    curr_output_directory=increment_dir_name(base_output_directory, run_base_name)
    if not os.path.exists(curr_output_directory):
        pathlib.Path(curr_output_directory).mkdir(parents=True, exist_ok=True)
    results_path = os.path.join(curr_output_directory, 'ae_iterate_beta_reconstruction.pickle')
    pickle.dump(all_results, open(results_path, 'wb'))
    
    M = Metadata()
    M.add_output(results_path)
    M.duration = stop - start
    metadata_path = os.path.join(curr_output_directory, 'ae_iterate_hidden_size_metadata.json')
    write_metadata(M, metadata_path)

Computing checksum for /mnt/smb/locker/issa-locker/users/Dan/code/ws/results/run741/ae_iterate_beta_reconstruction.pickle...
